In [1]:
import yfinance as yf
import pandas as pd
import os
import openpyxl
import json
import numpy as np # Import numpy for np.nan conversion
from datetime import datetime as dt

In [2]:
def _convert_na_to_none(data):
    """
    Recursively converts pd.NA values within a dictionary or list to None.
    Also converts numpy.nan to None as json.dumps cannot serialize it.
    """
    if isinstance(data, dict):
        return {k: _convert_na_to_none(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [_convert_na_to_none(elem) for elem in data]
    elif pd.isna(data): # Checks for both pd.NA and np.nan
        return None
    elif isinstance(data, float) and np.isnan(data): # Explicitly handle numpy NaNs
        return None
    else:
        return data

def compare_tickers_financials(tickers):
    """
    Compares key financial metrics for a list of tickers over the last 5 fiscal years
    using the yfinance library.

    Args:
        tickers (list): A list of stock ticker symbols (e.g., ["MSFT", "AAPL"]).

    Returns:
        dict: A dictionary where keys are ticker symbols and values are dictionaries
              containing financial data. Each financial data dictionary has years as keys,
              and each year's value is another dictionary with the requested metrics.
              A 'Current Valuation' key is also included for the latest valuation data.
              Returns an empty dictionary if no data can be fetched for any ticker,
              or if an error occurs for a specific ticker, an 'Error' key will be present
              for that ticker. All pd.NA and np.nan values are converted to None for JSON serialization.
    """
    all_tickers_data = {}
    
    # Get the current year to ensure we only look at past fiscal years
    current_year_actual = dt.now().year

    for ticker_symbol in tickers:
        print(f"Fetching data for {ticker_symbol}...")
        ticker = yf.Ticker(ticker_symbol)
        print(ticker.info.get('shortName'))
        ticker_data = {}

        try:
            # Fetch various financial statements and info
            financials = ticker.financials
            balance_sheet = ticker.balance_sheet
            cashflow = ticker.cashflow
            info = ticker.info  # For current valuation metrics and shares outstanding

            # Identify the last 5 unique fiscal years available across all statements
            # yfinance returns most recent data first, so columns are typically sorted descending
            available_years_financials = []
            if not financials.empty:
                available_years_financials = sorted(list(financials.columns.year.unique()), reverse=True)
            
            available_years_bs = []
            if not balance_sheet.empty:
                available_years_bs = sorted(list(balance_sheet.columns.year.unique()), reverse=True)

            available_years_cf = []
            if not cashflow.empty:
                available_years_cf = sorted(list(cashflow.columns.year.unique()), reverse=True)

            # Combine all available years and get the most recent unique 5
            all_available_years = sorted(list(set(available_years_financials + available_years_bs + available_years_cf)), reverse=True)
            
            # Filter to include only years up to the actual current year and take up to 5
            years_to_compare = [year for year in all_available_years if year <= current_year_actual][:5]
            
            if not years_to_compare:
                print(f"No historical financial data found for {ticker_symbol}.")
                all_tickers_data[ticker_symbol] = {'Error': 'No historical financial data found.'}
                continue


            for year in years_to_compare:
                year_data = {}
                
                # Helper to get the latest financial statement column for a given year
                def get_latest_column_for_year(df, target_year):
                    if df.empty:
                        return None
                    # Filter columns (which are Timestamps) for the target year
                    matching_dates = [col for col in df.columns if col.year == target_year]
                    if matching_dates:
                        # Return the latest date for that year
                        return max(matching_dates)
                    return None

                # --- P&L Metrics (from ticker.financials) ---
                financials_col = get_latest_column_for_year(financials, year)
                if financials_col and financials_col in financials.columns:
                    year_data['Revenue'] = financials.loc['Total Revenue', financials_col] if 'Total Revenue' in financials.index else pd.NA
                    year_data['Gross Profit'] = financials.loc['Gross Profit', financials_col] if 'Gross Profit' in financials.index else pd.NA
                    
                    # Gross Margin %
                    if pd.notna(year_data['Gross Profit']) and pd.notna(year_data['Revenue']) and year_data['Revenue'] != 0:
                        year_data['Gross Margin %'] = (year_data['Gross Profit'] / year_data['Revenue'])
                    else:
                        year_data['Gross Margin %'] = pd.NA
                    
                    year_data['SG&A'] = financials.loc['Selling General And Administration', financials_col] if 'Selling General And Administration' in financials.index else pd.NA
                    year_data['EBITDA'] = financials.loc['EBITDA', financials_col] if 'EBITDA' in financials.index else pd.NA
                    year_data['EBIT'] = financials.loc['EBIT', financials_col] if 'EBIT' in financials.index else pd.NA
                    year_data['Operating Income'] = financials.loc['Operating Income', financials_col] if 'Operating Income' in financials.index else pd.NA
                    year_data['Net Income'] = financials.loc['Net Income', financials_col] if 'Net Income' in financials.index else pd.NA

                    
                    # EBITDA Margin %
                    if pd.notna(year_data['EBITDA']) and pd.notna(year_data['Revenue']) and year_data['Revenue'] != 0:
                        year_data['EBITDA Margin %'] = (year_data['EBITDA'] / year_data['Revenue'])
                    else:
                        year_data['EBITDA Margin %'] = pd.NA
                
                    # EBIT Margin %
                    if pd.notna(year_data['EBIT']) and pd.notna(year_data['Revenue']) and year_data['Revenue'] != 0:
                        year_data['EBIT Margin %'] = (year_data['EBIT'] / year_data['Revenue'])
                    else:
                        year_data['EBIT Margin %'] = pd.NA
                        
                    # Operating Margin %
                    if pd.notna(year_data['Operating Income']) and pd.notna(year_data['Revenue']) and year_data['Revenue'] != 0:
                        year_data['Operating Margin %'] = (year_data['Operating Income'] / year_data['Revenue'])
                    else:
                        year_data['Operating Margin %'] = pd.NA

                    # Operating Margin %
                    if pd.notna(year_data['Net Income']) and pd.notna(year_data['Revenue']) and year_data['Revenue'] != 0:
                        year_data['Net Margin %'] = (year_data['Net Income'] / year_data['Revenue'])
                    else:
                        year_data['Net Margin %'] = pd.NA

                    
                    year_data['Diluted EPS'] = financials.loc['Diluted EPS', financials_col] if 'Diluted EPS' in financials.index else pd.NA
                    # Basic Average Shares needed for DPS and Cash per share calculations
                    year_data['Basic Average Shares'] = financials.loc['Basic Average Shares', financials_col] if 'Basic Average Shares' in financials.index else pd.NA
                else:
                    # Initialize P&L metrics with NA if data is missing for the year
                    year_data.update({
                        'Revenue': pd.NA, 'Gross Profit': pd.NA, 'Gross Margin %': pd.NA,
                        'SG&A': pd.NA, 'Operating Income': pd.NA, 'Operating Margin %': pd.NA,
                        'EBITDA': pd.NA, 'EBITDA Margin %': pd.NA, 'EBIT': pd.NA, 'EBIT Margin %': pd.NA,
                        'Diluted EPS': pd.NA, 'Basic Average Shares': pd.NA
                    })

                # --- Balance Sheet Metrics (from ticker.balance_sheet) ---
                balance_sheet_col = get_latest_column_for_year(balance_sheet, year)
                if balance_sheet_col and balance_sheet_col in balance_sheet.columns:
                    year_data['Total Assets'] = balance_sheet.loc['Total Assets', balance_sheet_col] if 'Total Assets' in balance_sheet.index else pd.NA
                    year_data['Total Liabilities Net Minority Interest'] = balance_sheet.loc['Total Liabilities Net Minority Interest', balance_sheet_col] if 'Total Liabilities Net Minority Interest' in balance_sheet.index else pd.NA
                    
                    # Get Total Debt
                    long_term_debt = balance_sheet.loc['Long Term Debt', balance_sheet_col] if 'Long Term Debt' in balance_sheet.index else pd.NA
                    
                    # Try common names for short term debt
                    short_term_debt = pd.NA
                    if 'Current Debt And Capital Lease Obligation' in balance_sheet.index:
                        short_term_debt = balance_sheet.loc['Current Debt And Capital Lease Obligation', balance_sheet_col]
                    
                    if pd.isna(short_term_debt) and 'Current Debt' in balance_sheet.index:
                        short_term_debt = balance_sheet.loc['Current Debt', balance_sheet_col]

                    total_debt = (long_term_debt if pd.notna(long_term_debt) else 0) + \
                                 (short_term_debt if pd.notna(short_term_debt) else 0)
                    
                    year_data['Total Debt'] = total_debt

                    total_equity = balance_sheet.loc['Total Equity Gross Minority Interest', balance_sheet_col] if 'Total Equity Gross Minority Interest' in balance_sheet.index else pd.NA
                    year_data['Total Equity Gross Minority Interest'] = total_equity
                    
                    # Debt to Equity
                    if pd.notna(total_equity) and total_equity != 0:
                        year_data['Debt to Equity'] = total_debt / total_equity
                    else:
                        year_data['Debt to Equity'] = pd.NA
                    
                    cash_and_equivalents = balance_sheet.loc['Cash And Cash Equivalents', balance_sheet_col] if 'Cash And Cash Equivalents' in balance_sheet.index else pd.NA
                    year_data['Cash And Cash Equivalents'] = cash_and_equivalents # Store for Cash per share calculation
                    
                    # Debt to Cash
                    if pd.notna(cash_and_equivalents) and cash_and_equivalents != 0:
                        year_data['Debt to Cash'] = total_debt / cash_and_equivalents
                    else:
                        year_data['Debt to Cash'] = pd.NA
                else:
                    # Initialize Balance Sheet metrics with NA
                    year_data.update({
                        'Total Assets': pd.NA, 'Total Liabilities Net Minority Interest': pd.NA,
                        'Total Debt': pd.NA, 'Total Equity Gross Minority Interest': pd.NA,
                        'Debt to Equity': pd.NA, 'Debt to Cash': pd.NA,
                        'Cash And Cash Equivalents': pd.NA
                    })

                # --- Cash Flow Metrics (from ticker.cashflow) ---
                cashflow_col = get_latest_column_for_year(cashflow, year)
                if cashflow_col and cashflow_col in cashflow.columns:
                    year_data['Operating Cash Flow'] = cashflow.loc['Operating Cash Flow', cashflow_col] if 'Operating Cash Flow' in cashflow.index else pd.NA
                    year_data['Investing Cash Flow'] = cashflow.loc['Investing Cash Flow', cashflow_col] if 'Investing Cash Flow' in cashflow.index else pd.NA
                    year_data['Financing Cash Flow'] = cashflow.loc['Financing Cash Flow', cashflow_col] if 'Financing Cash Flow' in cashflow.index else pd.NA
                    year_data['Dividend Payment'] = cashflow.loc['Cash Dividends Paid', cashflow_col] if 'Cash Dividends Paid' in cashflow.index else pd.NA
                else:
                    # Initialize Cash Flow metrics with NA
                    year_data.update({
                        'Operating Cash Flow': pd.NA, 'Investing Cash Flow': pd.NA,
                        'Financing Cash Flow': pd.NA, 'Dividend Payment': pd.NA
                    })
                
                # --- Valuation (Historical: DPS, Cash per share) ---
                # These rely on data from previous sections
                if pd.notna(year_data['Dividend Payment']) and pd.notna(year_data['Basic Average Shares']) and year_data['Basic Average Shares'] != 0:
                    year_data['DPS'] = year_data['Dividend Payment'] / year_data['Basic Average Shares']
                else:
                    year_data['DPS'] = pd.NA

                if pd.notna(year_data['Cash And Cash Equivalents']) and pd.notna(year_data['Basic Average Shares']) and year_data['Basic Average Shares'] != 0:
                    year_data['Cash per Share'] = year_data['Cash And Cash Equivalents'] / year_data['Basic Average Shares']
                else:
                    year_data['Cash per Share'] = pd.NA
                
                # Store the collected data for the current year
                ticker_data[year] = year_data

            # --- Current Valuation (from ticker.info) ---
            # These are current, point-in-time metrics.
            current_valuation = {}
            if info: # Ensure info is not empty/None
                current_valuation['Market Cap'] = info.get('marketCap')
                current_valuation['Price'] = info.get('currentPrice')
                current_valuation['52 Low'] = info.get('fiftyTwoWeekLow')
                current_valuation['52 Week High'] = info.get('fiftyTwoWeekHigh')
                current_valuation['PE'] = info.get('trailingPE')
                current_valuation['shortName']= info.get('shortName')
                current_valuation['currency'] = info.get('currency')
                
                # Fetch DPS (Current) and EPS (Current) for custom Payout Ratio calculation
                dps_current = info.get('dividendRate')
                eps_current = info.get('trailingEps')

                current_valuation['EPS (Current)'] = eps_current
                current_valuation['DPS (Current)'] = dps_current

                # Calculate Payout Ratio based on fetched DPS and EPS
                if pd.notna(dps_current) and pd.notna(eps_current) and eps_current != 0:
                    current_valuation['Payout Ratio'] = dps_current / eps_current
                else:
                    current_valuation['Payout Ratio'] = pd.NA # Or None, consistent with _convert_na_to_none

            if current_valuation:
                ticker_data['Current Valuation'] = current_valuation

            # Apply conversion before storing for the ticker
            all_tickers_data[ticker_symbol] = _convert_na_to_none(ticker_data)

        except Exception as e:
            print(f"Error fetching data for {ticker_symbol}: {e}")
            all_tickers_data[ticker_symbol] = {'Error': f"Could not retrieve data: {str(e)}"}
            # Ensure error message is also JSON serializable if it contains non-string types
            all_tickers_data[ticker_symbol] = _convert_na_to_none(all_tickers_data[ticker_symbol])

    return all_tickers_data

def export_financials_to_excel(financials_data, output_filename="financial_comparison.xlsx"):
    """
    Exports the financial comparison data to an Excel file.
    Each column in the Excel file represents a different ticker.

    Args:
        financials_data (dict): The output from the compare_tickers_financials function.
        output_filename (str): The name of the Excel file to create.
    """
    if not financials_data:
        print("No data provided to export to Excel.")
        return

    # Define the order of metrics for consistent output
    # Updated to include new P&L metrics and corrected Balance Sheet names
    metric_order = {
        "P&L": [
            'Revenue', 'Gross Profit', 'Gross Margin %', 'SG&A', 'Operating Income', 'Operating Margin %',
            'EBITDA', 'EBITDA Margin %', 'EBIT', 'EBIT Margin %', 'Net Income', 'Net Margin %',
            'Diluted EPS', 'Basic Average Shares'
        ],
        "Balance Sheet": [
            'Total Assets', 'Total Liabilities Net Minority Interest', 'Total Debt', 'Total Equity Gross Minority Interest',
            'Debt to Equity', 'Debt to Cash', 'Cash And Cash Equivalents'
        ],
        "Cash Flow": [
            'Operating Cash Flow', 'Investing Cash Flow', 'Financing Cash Flow', 'Dividend Payment'
        ],
        "Valuation (Historical)": [
            'DPS', 'Cash per Share'
        ],
        "Valuation (Current)": [
            'Market Cap', 'Price', '52 Low', '52 Week High', 'PE',
            'EPS (Current)', 'DPS (Current)', 'Payout Ratio','shortName','currency' # Payout Ratio is now calculated
        ]
    }

    # Collect all unique years from the data, excluding 'Current Valuation'
    all_years = set()
    for ticker_data in financials_data.values():
        for key in ticker_data.keys():
            if isinstance(key, int): # Check if key is a year (integer)
                all_years.add(key)
    sorted_years = sorted(list(all_years), reverse=True) # Sort years descending

    # Prepare data for DataFrame construction
    processed_data = {}

    for ticker_symbol, ticker_metrics in financials_data.items():
        if 'Error' in ticker_metrics:
            print(f"Skipping {ticker_symbol} due to error: {ticker_metrics['Error']}")
            continue

        for year_or_type, metrics_dict in ticker_metrics.items():
            if year_or_type == 'Current Valuation':
                category = "Valuation (Current)"
                for metric_name in metric_order[category]:
                    # For 'Payout Ratio', use the calculated value from compare_tickers_financials
                    value = metrics_dict.get(metric_name, None)
                    row_key = (category, metric_name, 'Current')
                    if row_key not in processed_data:
                        processed_data[row_key] = {}
                    processed_data[row_key][ticker_symbol] = value
            elif isinstance(year_or_type, int): # Historical data for a specific year
                year = year_or_type
                for category, metrics_list in metric_order.items():
                    if category == "Valuation (Current)": # Skip current valuation in this loop
                        continue
                    for metric_name in metrics_list:
                        value = metrics_dict.get(metric_name, None)
                        row_key = (category, metric_name, year)
                        if row_key not in processed_data:
                            processed_data[row_key] = {}
                        processed_data[row_key][ticker_symbol] = value

    # Create a DataFrame from the processed data
    df = pd.DataFrame.from_dict(processed_data, orient='index')

    # Assign meaningful names to the index levels
    df.index.names = ['Category', 'Metric', 'Year/Type']

    # Sort the index to ensure consistent order in the Excel file
    df = df.sort_index(level=[0, 1, 2], ascending=[True, True, False])

    # Reindex columns to ensure consistent ticker order if desired (optional)
    all_present_tickers = list(financials_data.keys())
    valid_tickers = [t for t in all_present_tickers if 'Error' not in financials_data.get(t, {})]
    if valid_tickers:
        df = df.reindex(columns=valid_tickers)

    # Write the DataFrame to an Excel file
    try:
        df.to_excel(output_filename)
        print(f"Financial data successfully exported to '{output_filename}'")
    except Exception as e:
        print(f"Error exporting data to Excel: {e}")



In [3]:
# --- Example Usage ---
# RESERVAS DE TICKERS
navieras = ["MAERSK-B.CO", "HLAG.DE","1919.HK","ZIM","TEN","EAT","SBLK","GSL","PSHG"] 
holdings_jp = ["8053.T","8031.T","8058.T","8002.T","8001.T"]
techs = ["MSFT","AMZN","AAPL","GOOG","NVDA","META","TSLA","AVGO"]
pharma = [
    "LLY",   # Eli Lilly
    "NVO",   # Novo Nordisk
    "JNJ",   # Johnson & Johnson
    "ABBV",  # AbbVie
    "RHHBY", # Roche (ADR)
    "NVS",   # Novartis
    "AZN",   # AstraZeneca
    "MRK",   # Merck & Co.
    "AMGN",  # Amgen
    "GILD",  # Gilead Sci.
    "PFE",   # Pfizer
    "VRTX",  # Vertex
    "BMY",   # Bristol-Myers Squibb
    "CSLLY", # CSL (Australia biotech)
    "REGN",  # Regeneron
    "ALNY",  # Alnylam
    "BAYRY", # Bayer (pharma wing)
    "TAK",   # Takeda
    "TEVA",  # Teva
    "BIIB"   # Biogen
]

sg_reits = [
    "C38U.SI",  # CapitaLand Integrated Commercial Trust
    "A17U.SI",  # CapitaLand Ascendas REIT
    "N2IU.SI",  # Mapletree Pan Asia Commercial Trust (MPACT)
    "M44U.SI",  # Mapletree Logistics Trust
    "ME8U.SI",  # Mapletree Industrial Trust
    "AJBU.SI",  # Keppel DC REIT
    "J69U.SI",  # Frasers Centrepoint Trust
    "C2PU.SI",  # Parkway Life REIT
    "AW9U.SI",  # First REIT
    "ACV.SI"    # Frasers Hospitality Trust
]

varios_tuit = ["AAF.L","CBOX.L","3679.T","3925.T","9435.T","5078.T","EVO.ST"] # https://x.com/not_macci/status/1941784811422859509/photo/1

# Define tickers
tickers_to_compare = sg_reits

# 1. Get the financial data
financials_data = compare_tickers_financials(tickers_to_compare)

# To pretty print the results (for verification in console, optional):
#print("\n--- Raw Financial Data (JSON format) ---")
#print(json.dumps(financials_data, indent=4))
#print("\n----------------------------------------")


# 2. Export the financial data to an Excel file
today = dt.today().strftime('%Y%m%d-%Hh%Mm')
path=fr"C:/Users/julia/Downloads/ticker_financial_comparison{today}.xlsx"
export_financials_to_excel(financials_data, path)

print(f"\nScript finished. Check '{path}' for the output.")

Fetching data for C38U.SI...
CapLand IntCom T
Fetching data for A17U.SI...
CapLand Ascendas REIT
Fetching data for N2IU.SI...
Mapletree PanAsia Com Tr
Fetching data for M44U.SI...
Mapletree Log Tr
Fetching data for ME8U.SI...
Mapletree Ind Tr
Fetching data for AJBU.SI...
Keppel DC Reit
Fetching data for J69U.SI...
FRASERS CENTREPOINT TRUST
Fetching data for C2PU.SI...
ParkwayLife Reit
Fetching data for AW9U.SI...
First Reit
Fetching data for ACV.SI...
FRASERS HOSPITALITY TRUST
Financial data successfully exported to 'C:/Users/julia/Downloads/ticker_financial_comparison20250807-17h58m.xlsx'

Script finished. Check 'C:/Users/julia/Downloads/ticker_financial_comparison20250807-17h58m.xlsx' for the output.


In [9]:
ftse100 = ['BATS.L','PSN.L','BTRW.L','ICG.L','SN.L','WTB.L','RR.L','IMI.L','BKG.L','STJ.L','AHT.L','SSE.L','BARC.L','SPX.L','AAF.L','PCT.L','SMIN.L','WEIR.L','NWG.L','LGEN.L','TW.L','IMB.L','CNA.L','GAW.L','MRO.L','CTEC.L','PSH.L','SGRO.L','HSBA.L','BT-A.L','CRDA.L','KGF.L','FCIT.L','HWDN.L','LAND.L','DCC.L','BP.L','CCEP.L','TSCO.L','MNG.L','SMT.L','AV.L','HLMA.L','VOD.L','PHNX.L','IAG.L','SHEL.L','UU.L','ITRK.L','SBRY.L','GSK.L','CCH.L','ENT.L','STAN.L','BNZL.L','SVT.L','ABF.L','JD.L','SDR.L','DPLM.L','IHG.L','DGE.L','UTG.L','SGE.L','INF.L','HSX.L','RMV.L','NG.L','LMP.L','ALW.L','AZN.L','MNDI.L','CPG.L','NXT.L','LLOY.L','REL.L','EXPN.L','AUTO.L','HIK.L','BEZ.L','HLN.L','PSON.L','MKS.L','BA.L','RIO.L','BAB.L','ULVR.L','III.L','LSEG.L','RKT.L','EDV.L','PRU.L','FRES.L','RTO.L','ADM.L','EZJ.L','ANTO.L','AAL.L','GLEN.L','WPP.L']